# Week 09: ML Reliability, Distribution Shift, and Error Analysis
Machine Learning in Engineering Science / NCKU + LAiMM / Fall 2026

**Class date:** Monday, 2 November 2026, 14:00-17:00. Week 8 is a scheduled holiday.

**Learning target:** diagnose a misspecified model, separate selection from final evaluation, and reproduce an evidence package from frozen data.

**Class rhythm:** H1: 45 minutes of lecture and 15 minutes of hand calculation; H2: 15 minutes of lecture and 45 minutes of guided practice; H3: 60 minutes of lab and explain-back. Total: 60 minutes of explanation and 120 minutes of practice, with eight core exercises and two optional challenges. It does not create a new weekly graded assignment.

**Cost and runtime:** a free Colab CPU runtime or Python 3 Jupyter. No API key, GPU, external dataset or finance download is needed. All observations are synthetic and all metrics must come from your actual run.

Run cells from top to bottom. Read the prediction question before each experiment. Do not replace an observed score with a number from the slides.

## 1. Establish the contract before fitting
One row is one independent synthetic observation. Inputs are `x0`, `x1`, and `x2`; they have no claimed physical units. The response has synthetic response units.

The known generating relation is `y = 2*x0 + x1**2 + noise`, with zero-mean Gaussian noise of standard deviation 0.2. The unused third feature is a deliberate negative control.

We predeclare two candidates: a linear Ridge pipeline and a Ridge pipeline with an additional `x1**2` feature. Choose between them using validation MAE only. Both candidates and all stress conditions are defined before final test results are inspected.

The residual convention in this week is **prediction minus observation**. A positive residual means overprediction. Always read a plot's sign convention.

In [ ]:
import os, sys, subprocess, importlib.util, platform, json, csv, hashlib, zipfile
from pathlib import Path
for variable in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ.setdefault(variable, '1')
required = {'numpy':'numpy', 'matplotlib':'matplotlib', 'sklearn':'scikit-learn'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error
CONFIG = {'seed':2026, 'n':500, 'noise_sd':0.2, 'alpha':1.0,
          'subgroup_boundary':1.0, 'stress_n':150, 'stress_scales':[1.0,1.5,2.5],
          'bootstrap_resamples':300, 'selection':'validation MAE; tie favors linear'}
RUN_LABEL = 'reference'  # Give a controlled extension a different label.
config_id = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:10]
ARTIFACTS = Path('artifacts') / ('week09-' + RUN_LABEL + '-' + config_id)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
ENV = {'python':platform.python_version(), 'numpy':np.__version__,
       'scikit_learn':sklearn.__version__, 'matplotlib':matplotlib.__version__}
plt.rcParams.update({'figure.figsize':(10,6.9), 'figure.dpi':100, 'savefig.dpi':160,
                     'font.size':22, 'axes.labelsize':22, 'axes.titlesize':24,
                     'xtick.labelsize':18, 'ytick.labelsize':18, 'legend.fontsize':17,
                     'axes.spines.top':False, 'axes.spines.right':False})
COLORS = {'linear':'#7f2635', 'quadratic':'#167b87', 'gold':'#b88a29'}
def finish_figure(fig, name):
    fig.savefig(ARTIFACTS/name, bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
print('Environment:', ENV)
print('Configuration:', CONFIG)
print('Evidence folder:', ARTIFACTS)

**Checkpoint:** record the actual versions printed above. A seed is one part of reproducibility, not a complete experiment identity. We also preserve the exact CSV and row-to-split mapping.

## 2. Generate data and reserve final evidence
The split is 350 training, 75 validation and 75 test rows. A second random generator supplies a separate stress population later. Training, validation and test IDs must be disjoint.

**Predict:** does a random-row split answer a new-machine or future-time question? Here rows are independent synthetic observations. Real repeated machines or temporal records require different splitting rules.

In [ ]:
rng_data = np.random.default_rng(CONFIG['seed'])
X = rng_data.normal(size=(CONFIG['n'],3))
noise = rng_data.normal(0,CONFIG['noise_sd'],CONFIG['n'])
y = 2*X[:,0] + X[:,1]**2 + noise
ids = np.arange(len(X))
train_idx, held_idx = train_test_split(ids,test_size=.30,random_state=CONFIG['seed'])
val_idx, test_idx = train_test_split(held_idx,test_size=.50,random_state=CONFIG['seed']+1)
train_idx, val_idx, test_idx = map(np.sort,(train_idx,val_idx,test_idx))
assert (len(train_idx),len(val_idx),len(test_idx)) == (350,75,75)
assert not (set(train_idx)&set(val_idx) or set(train_idx)&set(test_idx) or set(val_idx)&set(test_idx))
assert set(np.r_[train_idx,val_idx,test_idx]) == set(ids)
Xtr,ytr = X[train_idx],y[train_idx]
Xva,yva = X[val_idx],y[val_idx]
Xte,yte = X[test_idx],y[test_idx]
print('Shapes:', Xtr.shape, Xva.shape, Xte.shape)
print('Target rule: 2*x0 + x1**2 + Gaussian noise')

## 3. Freeze identities and measurements
The CSV stores the source row ID, split membership, every input and the observed target. A SHA-256 identifies these exact file bytes. The checksum detects file changes; it does not prove that the data or scientific claim is valid.

**Action:** open the exported CSV in the Colab Files panel after this cell. Do not edit it to improve a result.

In [ ]:
split = np.full(len(X),'train',dtype='<U5');split[val_idx]='val';split[test_idx]='test'
data_path = ARTIFACTS/'week09-frozen-data.csv'
with data_path.open('w',newline='',encoding='utf-8') as handle:
    writer=csv.writer(handle);writer.writerow(['sample_id','split','x0','x1','x2','y'])
    for i in ids:
        writer.writerow([f'row-{i:04d}',split[i],*[repr(float(v)) for v in X[i]],repr(float(y[i]))])
data_sha256=hashlib.sha256(data_path.read_bytes()).hexdigest()
print('Frozen CSV:',data_path.name,'SHA-256:',data_sha256)
print('First training IDs:',[f'row-{i:04d}' for i in train_idx[:5]])

## 4. Predefine and fit both candidates
The first pipeline cannot exactly represent the squared term. The second explicitly adds that term. Both fit their scalers only on training inputs and use the same Ridge penalty, `alpha=1`.

`FunctionTransformer` applies a fixed, label-free mapping. It does not discover the quadratic relation or fit it from test labels. The known synthetic relation lets us predeclare this comparison honestly.

**Predict:** which candidate should reduce structured residual curvature? State a mechanism rather than promising a score.

In [ ]:
def append_square(A):
    A=np.asarray(A,dtype=float)
    return np.column_stack([A,A[:,1]**2])
models={
    'linear':make_pipeline(StandardScaler(),Ridge(alpha=CONFIG['alpha'],solver='svd')),
    'quadratic':make_pipeline(FunctionTransformer(append_square,validate=True),
                             StandardScaler(),Ridge(alpha=CONFIG['alpha'],solver='svd'))
}
for model in models.values():
    model.fit(Xtr,ytr)

def metric_row(actual,predicted):
    return {'n':int(len(actual)), 'MAE':float(mean_absolute_error(actual,predicted)),
            'RMSE':float(np.sqrt(mean_squared_error(actual,predicted)))}
validation={name:metric_row(yva,model.predict(Xva)) for name,model in models.items()}
selected=min(models, key=lambda name:(validation[name]['MAE'],0 if name=='linear' else 1))
print('Validation results:',json.dumps(validation,indent=2))
print('Frozen selected candidate:',selected)
fig,ax=plt.subplots(layout='constrained')
ax.bar(list(validation),[validation[n]['MAE'] for n in validation],color=[COLORS[n] for n in validation])
ax.set(xlabel='Predeclared candidate',ylabel='Validation MAE (response units)')
finish_figure(fig,'w09-validation-selection.png')

**Checkpoint:** the selection rule has now frozen a candidate. Do not switch the winner after seeing final test errors. Both candidates may be reported on test as a predeclared comparison, but the validation-selected candidate remains the selected one.

## 5. Evaluate the final test and define slices
Compute MAE and RMSE for each frozen candidate. The subgroup rule `abs(x1)>1` is also predeclared. Always report group counts. A small or empty group does not justify a confident general conclusion.

In [ ]:
test_prediction={name:model.predict(Xte) for name,model in models.items()}
test_metrics={name:metric_row(yte,p) for name,p in test_prediction.items()}
extreme=np.abs(Xte[:,1])>CONFIG['subgroup_boundary']
slice_masks={'ordinary':~extreme,'extreme':extreme}
slices={}
for name,prediction in test_prediction.items():
    slices[name]={}
    for group,mask in slice_masks.items():
        slices[name][group]=metric_row(yte[mask],prediction[mask]) if mask.any() else {'n':0,'MAE':None,'RMSE':None}
print('Final test:',json.dumps(test_metrics,indent=2))
print('Predeclared slices:',json.dumps(slices,indent=2))
for name in models:
    weighted=sum(slices[name][g]['n']*slices[name][g]['MAE'] for g in slices[name] if slices[name][g]['n'])/len(yte)
    np.testing.assert_allclose(weighted,test_metrics[name]['MAE'],rtol=1e-12)
fig,ax=plt.subplots(layout='constrained');positions=np.arange(2);width=.34
for j,metric in enumerate(['MAE','RMSE']):
    ax.bar(positions+(j-.5)*width,[test_metrics[n][metric] for n in models],width,label=metric)
ax.set_xticks(positions,list(models));ax.set_ylabel('Response units');ax.legend()
finish_figure(fig,'w09-test-metrics.png')
fig,ax=plt.subplots(layout='constrained')
for j,name in enumerate(models):
    vals=[slices[name][g]['MAE'] for g in slice_masks]
    ax.bar(np.arange(2)+(j-.5)*width,vals,width,label=name,color=COLORS[name])
ax.set_xticks([0,1],[g+' (n='+str(int(m.sum()))+')' for g,m in slice_masks.items()])
ax.set_ylabel('Test MAE (response units)');ax.legend()
finish_figure(fig,'w09-test-slices.png')

## 6. Inspect residuals instead of only a mean
For this week `residual = prediction - observation`. Negative residuals indicate underprediction. Plot residual against the feature containing the missing squared term.

**Predict:** a linear approximation to a quadratic response can leave an inverted-U residual pattern. Does the actual plot show that pattern? In real data a pattern is a diagnostic clue, not proof of a particular physical cause.

In [ ]:
for name,prediction in test_prediction.items():
    residual=prediction-yte
    fig,ax=plt.subplots(layout='constrained');ax.scatter(Xte[:,1],residual,c=COLORS[name],s=44,alpha=.8)
    ax.axhline(0,color='black',lw=1);ax.axvline(-1,color='gray',ls=':');ax.axvline(1,color='gray',ls=':')
    ax.set(xlabel='x1 (synthetic input)',ylabel='Prediction - observation',title=name+' / held-out test')
    finish_figure(fig,'w09-'+name+'-residuals.png')
fig,ax=plt.subplots(layout='constrained')
for name,p in test_prediction.items():ax.scatter(yte,p,s=42,alpha=.65,label=name,color=COLORS[name])
limits=[float(min(yte.min(),min(p.min() for p in test_prediction.values()))),float(max(yte.max(),max(p.max() for p in test_prediction.values())))]
ax.plot(limits,limits,'k--',lw=1);ax.set(xlabel='Observed response',ylabel='Predicted response');ax.legend()
finish_figure(fig,'w09-test-parity.png')
fig,ax=plt.subplots(layout='constrained');ax.scatter(Xtr[:,1],ytr-2*Xtr[:,0],s=20,alpha=.5,color=COLORS['quadratic'])
grid=np.linspace(Xtr[:,1].min(),Xtr[:,1].max(),200);ax.plot(grid,grid**2,color=COLORS['linear'],lw=2,label='Known generating term')
ax.set(xlabel='Training x1',ylabel='y - 2*x0');ax.legend()
finish_figure(fig,'w09-known-mechanism.png')

## 7. Stress one input distribution without changing the mechanism
Generate a separate 150-row population. Across the three conditions, keep the same base observations, `x0`, `x2` and noise. Multiply only `x1` by 1, 1.5 or 2.5, then recompute targets from the same generating relation.

This is a controlled **covariate shift** experiment: the distribution of inputs changes while the stipulated conditional target mechanism stays the same. It is not a model of every deployment failure and it does not establish that all shifts cause a particular amount of error.

**Predict:** should a model that represents the squared term respond differently from a misspecified linear model? Keep all fitted parameters frozen.

In [ ]:
rng_stress=np.random.default_rng(CONFIG['seed']+100)
base_stress=rng_stress.normal(size=(CONFIG['stress_n'],3))
stress_noise=rng_stress.normal(0,CONFIG['noise_sd'],CONFIG['stress_n'])
def parameter_snapshot(model):
    reg=model.named_steps['ridge'];scale=model.named_steps['standardscaler']
    return np.r_[reg.coef_,reg.intercept_,scale.mean_,scale.scale_].copy()
parameters_before={name:parameter_snapshot(model) for name,model in models.items()}
stress_results=[];stress_predictions={};stress_inputs={}
for severity in CONFIG['stress_scales']:
    A=base_stress.copy();A[:,1]*=severity
    target=2*A[:,0]+A[:,1]**2+stress_noise
    stress_inputs[severity]=(A,target)
    stress_predictions[severity]={}
    for name,model in models.items():
        pred=model.predict(A);stress_predictions[severity][name]=pred
        stress_results.append({'scale':severity,'model':name,**metric_row(target,pred)})
for name,model in models.items():np.testing.assert_array_equal(parameters_before[name],parameter_snapshot(model))
print(json.dumps(stress_results,indent=2))
fig,ax=plt.subplots(layout='constrained')
for name in models:
    records=[r for r in stress_results if r['model']==name]
    ax.plot([r['scale'] for r in records],[r['MAE'] for r in records],'o-',label=name,color=COLORS[name],lw=2)
ax.set(xlabel='Multiplier on x1',ylabel='Paired stress-population MAE');ax.legend()
finish_figure(fig,'w09-stress-mae.png')
fig,ax=plt.subplots(layout='constrained')
ax.hist(Xtr[:,1],bins=18,density=True,histtype='step',lw=2,label='training x1',color=COLORS['quadratic'])
ax.hist(stress_inputs[2.5][0][:,1],bins=18,density=True,histtype='step',lw=2,label='stress x1 (scale 2.5)',color=COLORS['linear'])
ax.set(xlabel='x1',ylabel='Empirical density');ax.legend()
finish_figure(fig,'w09-input-shift.png')

## 8. Check tails, ranges and population sizes
A value outside the observed training range is a range flag, not mathematical proof that it lies outside a Gaussian distribution's support. Gaussian support is unbounded.

A changed input histogram can motivate investigation even before new labels arrive. It cannot, by itself, measure prediction error. Error assessment requires outcomes or another justified evaluation procedure.

In [ ]:
A_stress,y_stress=stress_inputs[2.5]
fig,ax=plt.subplots(layout='constrained')
for name,p in stress_predictions[2.5].items():
    ax.scatter(y_stress,p,s=32,alpha=.65,label=name,color=COLORS[name])
low=min(y_stress.min(),min(p.min() for p in stress_predictions[2.5].values()))
high=max(y_stress.max(),max(p.max() for p in stress_predictions[2.5].values()))
ax.plot([low,high],[low,high],'k--',lw=1);ax.set(xlabel='Stress observed response',ylabel='Frozen model prediction');ax.legend()
finish_figure(fig,'w09-stress-parity.png')
fig,ax=plt.subplots(layout='constrained')
for name,p in stress_predictions[2.5].items():
    e=np.sort(np.abs(p-y_stress));ax.plot(e,np.arange(1,len(e)+1)/len(e),label=name,color=COLORS[name],lw=2)
ax.set(xlabel='Absolute response error',ylabel='Empirical cumulative fraction');ax.legend()
finish_figure(fig,'w09-stress-error-cdf.png')
training_range=[float(Xtr[:,1].min()),float(Xtr[:,1].max())]
range_flag=(A_stress[:,1]<training_range[0])|(A_stress[:,1]>training_range[1])
fig,ax=plt.subplots(layout='constrained');ax.scatter(A_stress[:,1],np.abs(stress_predictions[2.5]['linear']-y_stress),c=np.where(range_flag,COLORS['linear'],COLORS['quadratic']),s=34)
for boundary in training_range:ax.axvline(boundary,color='black',ls='--',lw=1)
ax.set(xlabel='Stress x1; dashed = observed training range',ylabel='Linear-model absolute error')
finish_figure(fig,'w09-observed-range.png')
print('Training x1 range:',training_range,'Stress rows outside it:',int(range_flag.sum()),'/',len(A_stress))

## 9. Add a bounded uncertainty description
A bootstrap resamples test residual magnitudes for a fixed fitted model. Here we report a descriptive percentile interval for subgroup MAE, conditional on this model and the IID-within-slice assumption.

This interval does not include every source of training, model-selection or deployment-shift uncertainty. With fewer than two observations we report no interval. A small or narrow interval cannot prove safety under a new population.

In [ ]:
def bootstrap_mae_interval(errors,seed):
    errors=np.asarray(errors,dtype=float)
    if len(errors)<2:return None
    rng=np.random.default_rng(seed)
    means=errors[rng.integers(0,len(errors),(CONFIG['bootstrap_resamples'],len(errors)))].mean(axis=1)
    return [float(v) for v in np.percentile(means,[2.5,97.5])]
intervals={}
for j,(group,mask) in enumerate(slice_masks.items()):
    errors=np.abs(test_prediction[selected][mask]-yte[mask])
    intervals[group]={'n':int(mask.sum()),'interval':bootstrap_mae_interval(errors,CONFIG['seed']+200+j)}
print('Descriptive fixed-model bootstrap:',json.dumps(intervals,indent=2))
fig,ax=plt.subplots(layout='constrained')
for j,group in enumerate(slice_masks):
    item=intervals[group];center=slices[selected][group]['MAE']
    if center is not None:
        ax.plot(j,center,'o',color=COLORS[selected],ms=9)
        if item['interval'] is not None:ax.vlines(j,*item['interval'],color=COLORS[selected],lw=3)
ax.set_xticks([0,1],[g+' (n='+str(intervals[g]['n'])+')' for g in slice_masks])
ax.set_ylabel('Test MAE; descriptive bootstrap interval')
finish_figure(fig,'w09-slice-intervals.png')

## 10. Recover an interpretable prediction equation
A coefficient after standardization is not directly in the original feature scale. Undo scaling before comparing with the known synthetic coefficients. The original-space intercept is `intercept_scaled - sum(coef_scaled*mean/scale)`.

This is an exact description of the fitted model's function. It is not a causal discovery about a real engineering system.

In [ ]:
coefficient_record={}
for name,model in models.items():
    scaler=model.named_steps['standardscaler'];reg=model.named_steps['ridge']
    raw=reg.coef_/scaler.scale_
    intercept=float(reg.intercept_-np.dot(raw,scaler.mean_))
    transformed=append_square(Xte) if name=='quadratic' else Xte
    explicit=intercept+transformed@raw
    np.testing.assert_allclose(explicit,test_prediction[name],atol=1e-10,rtol=1e-10)
    coefficient_record[name]={'features':['x0','x1','x2']+(['x1_squared'] if name=='quadratic' else []),
                              'raw_coefficients':raw.tolist(),'raw_intercept':intercept,
                              'scaler_mean':scaler.mean_.tolist(),'scaler_scale':scaler.scale_.tolist()}
print(json.dumps(coefficient_record,indent=2))
fig,ax=plt.subplots(layout='constrained');xx=np.arange(4)
ax.bar(xx-.17,[2,0,0,1],.34,label='Known generating coefficients',color=COLORS['gold'])
ax.bar(xx+.17,coefficient_record['quadratic']['raw_coefficients'],.34,label='Fitted quadratic model',color=COLORS['quadratic'])
ax.axhline(0,color='black',lw=1);ax.set_xticks(xx,['x0','x1','x2','x1 squared']);ax.set_ylabel('Original-space coefficient');ax.legend()
finish_figure(fig,'w09-coefficients.png')

## 11. Reproduce from the CSV rather than hidden memory
Load the exact frozen file, recover its training and test rows, refit a clone of the already selected candidate and compare predictions. This does not reselect the model using test labels.

**Checkpoint:** a faithful rerun should agree within the stated numerical tolerance. If it does not, compare file hash, feature order, split IDs, configuration and versions before telling a story about a changed score.

In [ ]:
with data_path.open(newline='',encoding='utf-8') as handle: frozen=list(csv.DictReader(handle))
X_re=np.array([[float(row[c]) for c in ['x0','x1','x2']] for row in frozen])
y_re=np.array([float(row['y']) for row in frozen]);split_re=np.array([row['split'] for row in frozen])
replayed=clone(models[selected]).fit(X_re[split_re=='train'],y_re[split_re=='train'])
replayed_pred=replayed.predict(X_re[split_re=='test'])
max_replay_difference=float(np.max(np.abs(replayed_pred-test_prediction[selected])))
np.testing.assert_allclose(replayed_pred,test_prediction[selected],atol=1e-10,rtol=1e-10)
assert hashlib.sha256(data_path.read_bytes()).hexdigest()==data_sha256
print('Replayed selected model:',selected,'Maximum prediction difference:',max_replay_difference)
print('Frozen data hash unchanged:',data_sha256)

## 12. Export an auditable result package
The result includes actual metrics, row counts, candidate definitions, selection rule, original-space coefficients, data checksum, runtime versions and the replay check. Prediction CSV rows identify the cohort and observation.

The ZIP is a portable teaching record, not proof of deployment reliability. Download it before ending Colab. Keep a separate run label when making a controlled comparison.

In [ ]:
prediction_path=ARTIFACTS/'week09-predictions.csv'
with prediction_path.open('w',newline='',encoding='utf-8') as handle:
    writer=csv.writer(handle);writer.writerow(['cohort','sample_id','model','x1','observed','predicted','residual'])
    for name,p in test_prediction.items():
        for j,i in enumerate(test_idx):writer.writerow(['test',f'row-{i:04d}',name,repr(float(Xte[j,1])),repr(float(yte[j])),repr(float(p[j])),repr(float(p[j]-yte[j]))])
    for severity,(A,target) in stress_inputs.items():
        for name,p in stress_predictions[severity].items():
            for j in range(len(A)):writer.writerow([f'stress-{severity}',f'stress-{j:04d}',name,repr(float(A[j,1])),repr(float(target[j])),repr(float(p[j])),repr(float(p[j]-target[j]))])
record={'week':9,'data_scope':'synthetic teaching experiment','config':CONFIG,'environment':ENV,
        'data_sha256':data_sha256,'split_counts':{'train':350,'validation':75,'test':75},
        'residual_definition':'prediction minus observation','validation':validation,
        'selected_on_validation':selected,'test':test_metrics,'slices':slices,'stress':stress_results,
        'observed_training_x1_range':training_range,'stress_outside_observed_range':int(range_flag.sum()),
        'fixed_model_bootstrap':intervals,'coefficients':coefficient_record,
        'replay_max_prediction_difference':max_replay_difference,
        'claim_limit':'Predeclared synthetic populations; not real-world deployment validation.'}
evidence_path=ARTIFACTS/'week09-evidence.json'
evidence_path.write_text(json.dumps(record,indent=2,allow_nan=False),encoding='utf-8')
(ARTIFACTS/'week09-coefficients.json').write_text(json.dumps(coefficient_record,indent=2),encoding='utf-8')
(ARTIFACTS/'RESULT-CARD-TEMPLATE.md').write_text(chr(10).join(['# Result card','','Claim:','Evidence:','Limitation:','Next test:','AI use and checks (or none):','']),encoding='utf-8')
BUNDLE=ARTIFACTS/'week09-reliability-evidence.zip'
with zipfile.ZipFile(BUNDLE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in sorted(ARTIFACTS.iterdir()):
        if file.is_file() and file.suffix!='.zip' and (file.name.startswith(('w09-','week09-')) or file.name=='RESULT-CARD-TEMPLATE.md'):z.write(file,arcname=file.name)
with zipfile.ZipFile(BUNDLE) as z:assert z.testzip() is None
print('Saved:',evidence_path)
print('Bundle:',BUNDLE,'Bytes:',BUNDLE.stat().st_size)
print('Use Colab Files: artifacts -> this run folder -> right-click the ZIP -> Download.')

## 13. Explain before making another change
Complete these in your own words:

1. What relationship did the linear model omit, and which actual residual plot supports your answer?
2. What changed in the stress population and what remained fixed?
3. Which metric hides a subgroup failure if reported alone?
4. Why is the validation-selected model still the selected model after the final test?
5. What does the exact CSV replay establish, and what does it not establish?

**One controlled extension:** predeclare a new stress multiplier and a new run label. Keep the candidate fits and the original reference evidence. An unplanned final-test-driven change is exploration and needs fresh final evaluation before a new performance claim.

**Interpretability boundary:** the synthetic generator makes a quadratic mechanism known. In real observational data, a curved residual plot suggests misspecification but does not prove a physical cause.

### References
- [Google ML Crash Course: monitoring pipelines](https://developers.google.com/machine-learning/crash-course/production-ml-systems/monitoring)
- [WILDS research paper](https://proceedings.mlr.press/v139/koh21a.html)
- [scikit-learn: common pitfalls and leakage-safe pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- [scikit-learn: regression evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)